In [1]:
from pathlib import Path
import pandas as pd

# ---------------------------
# User-configurable paths
# ---------------------------
# manifest_csv = Path("table/runs_manifest_20260409_042243.csv")
manifest_csv = Path("table/runs_manifest_20260421_141025_2.csv")
evals_root = Path("evals")

# ---------------------------
# Table config
# ---------------------------
CONFIG_ORDER = ["CRW", "EE", "POI", "LPOI"]

MODEL_LABELS = {
    "rule-based": "Rule-based",
    "ppo": "PPO",
    "sac": "SAC",
    "dqn": "DQN",
}

MODEL_ORDER = ["rule-based", "dqn", "ppo", "sac"]
LEARNED_MODELS = ["dqn", "ppo", "sac"]

# (output_key, latex_label, source_column, transform_fn)
METRICS = [
    ("reward", "Total reward", "reward", lambda s: s),
    ("monitoring_reward", "Monitoring reward", "r_dist", lambda s: s + 1),
    ("disturbance_penalty", "Disturbance penalty", "p_disturbance", lambda s: s),
]

AGENT_CSVS = {
    "ppo": "ppo_ep.csv",
    "sac": "sac_ep.csv",
    "dqn": "dqn_ep.csv",
}

RULE_BASED_CSV = "CentroidStandoff_ep.csv"

# ---------------------------
# Helpers
# ---------------------------
def summarize_series(series: pd.Series) -> dict:
    mean = series.mean()
    std = series.std(ddof=1)
    return {
        "mean": mean,
        "std": std,
        "formatted": f"${mean:.3f} \\pm {std:.3f}$",
    }

def fmt_mean_only(x: float) -> str:
    return f"${x:.3f}$"

def read_eval_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, comment="#")
    required_cols = [source_col for _, _, source_col, _ in METRICS]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} is missing required columns: {missing}")
    return df

def summarize_df(df: pd.DataFrame) -> dict:
    return {
        output_key: summarize_series(transform_fn(df[source_col]))
        for output_key, _, source_col, transform_fn in METRICS
    }

# ---------------------------
# Load manifest
# ---------------------------
manifest = pd.read_csv(manifest_csv)

required_manifest_cols = {"config", "agent", "eval_name"}
missing_manifest_cols = required_manifest_cols - set(manifest.columns)
if missing_manifest_cols:
    raise ValueError(f"Manifest is missing required columns: {sorted(missing_manifest_cols)}")

# ---------------------------
# Aggregate results
# ---------------------------
results = {}

# PPO / SAC / DQN from agent-specific files
for _, row in manifest.iterrows():
    config = str(row["config"]).strip()
    agent = str(row["agent"]).strip().lower()
    eval_name = str(row["eval_name"]).strip()

    if agent not in AGENT_CSVS:
        raise ValueError(f"Unknown agent type: {agent}")

    csv_path = evals_root / eval_name / AGENT_CSVS[agent]
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing eval CSV for {agent}: {csv_path}")

    df = read_eval_csv(csv_path)

    results.setdefault(agent, {})
    results[agent][config] = summarize_df(df)

# Rule-based from CentroidStandoff_ep.csv
results["rule-based"] = {}

for config in CONFIG_ORDER:
    matching = manifest[manifest["config"].astype(str).str.strip() == config]
    if matching.empty:
        continue

    eval_name = str(matching.iloc[0]["eval_name"]).strip()
    csv_path = evals_root / eval_name / RULE_BASED_CSV

    if not csv_path.exists():
        raise FileNotFoundError(f"Missing rule-based CSV for {config}: {csv_path}")

    df = read_eval_csv(csv_path)
    results["rule-based"][config] = summarize_df(df)

# ---------------------------
# Compute learned-method averages
# ---------------------------
learned_avg = {}

for output_key, _, _, _ in METRICS:
    learned_avg[output_key] = {}

    for config in CONFIG_ORDER:
        vals = []
        for model_key in LEARNED_MODELS:
            if model_key in results and config in results[model_key]:
                vals.append(results[model_key][config][output_key]["mean"])

        learned_avg[output_key][config] = fmt_mean_only(sum(vals) / len(vals)) if vals else "--"

# ---------------------------
# Build LaTeX table
# ---------------------------
lines = []
lines.append(r"\begin{table}[!ht]")
lines.append(r"\centering")
lines.append(r"\setlength{\tabcolsep}{4pt}")
lines.append(
    r"\caption{Performance comparison across behaviors. Results are reported as mean $\pm$ standard deviation across evaluation runs of \(N=100\) repetitions. The final block reports the average mean across learned methods (DQN, PPO, and SAC).}"
)
lines.append(r"\label{tab:behavior_results_full}")
lines.append(r"\begin{tabular}{llcccc}")
lines.append(r"\hline \hline")
lines.append(r"\textbf{Model} & \textbf{Metric} & \textbf{CRW} & \textbf{EE} & \textbf{POI} & \textbf{LPOI} \\")
lines.append(r"\hline \hline")

models_present = [m for m in MODEL_ORDER if m in results and results[m]]

for mi, model_key in enumerate(models_present):
    model_block = results[model_key]
    model_label = MODEL_LABELS[model_key]

    for metric_idx, (output_key, metric_label, _, _) in enumerate(METRICS):
        row_cells = []
        row_cells.append(model_label if metric_idx == 0 else "")
        row_cells.append(metric_label)

        for config in CONFIG_ORDER:
            value = model_block.get(config, {}).get(output_key, {}).get("formatted", "--")
            row_cells.append(value)

        lines.append(" & ".join(row_cells) + r" \\")
        lines.append("")

    if mi < len(models_present) - 1:
        lines.append(r"\midrule")

# Final learned-average block
lines.append(r"\midrule")
for metric_idx, (output_key, metric_label, _, _) in enumerate(METRICS):
    row_cells = []
    row_cells.append(r"\textit{Avg. learned}" if metric_idx == 0 else "")
    row_cells.append(metric_label)

    for config in CONFIG_ORDER:
        row_cells.append(learned_avg[output_key][config])

    lines.append(" & ".join(row_cells) + r" \\")
    lines.append("")

lines.append(r"\hline \hline")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)
print(latex_table)

\begin{table}[!ht]
\centering
\setlength{\tabcolsep}{4pt}
\caption{Performance comparison across behaviors. Results are reported as mean $\pm$ standard deviation across evaluation runs of \(N=100\) repetitions. The final block reports the average mean across learned methods (DQN, PPO, and SAC).}
\label{tab:behavior_results_full}
\begin{tabular}{llcccc}
\hline \hline
\textbf{Model} & \textbf{Metric} & \textbf{CRW} & \textbf{EE} & \textbf{POI} & \textbf{LPOI} \\
\hline \hline
Rule-based & Total reward & $0.772 \pm 0.024$ & $0.825 \pm 0.022$ & $0.771 \pm 0.022$ & $0.774 \pm 0.018$ \\

 & Monitoring reward & $0.485 \pm 0.006$ & $0.497 \pm 0.004$ & $0.483 \pm 0.007$ & $0.474 \pm 0.007$ \\

 & Disturbance penalty & $0.217 \pm 0.008$ & $0.198 \pm 0.008$ & $0.215 \pm 0.007$ & $0.207 \pm 0.004$ \\

\midrule
DQN & Total reward & $0.909 \pm 0.006$ & $0.886 \pm 0.009$ & $0.916 \pm 0.010$ & $0.855 \pm 0.048$ \\

 & Monitoring reward & $0.585 \pm 0.018$ & $0.546 \pm 0.013$ & $0.554 \pm 0.010$ & $0.5

In [2]:
from pathlib import Path
import re
import pandas as pd

# ---------------------------
# User-configurable paths
# ---------------------------
ROOT_CANDIDATES = [
    Path("evals/eval_animals_new_lpoi")
]

# ---------------------------
# Table config
# ---------------------------
CONFIG_ORDER = ["CRW", "EE", "POI", "LPOI"]

MODEL_LABELS = {
    "rule-based": "Rule-based",
    "dqn": "DQN",
    "ppo": "PPO",
    "sac": "SAC",
}
MODEL_ORDER = ["rule-based", "dqn", "ppo", "sac"]

ANIMAL_DISPLAY = {
    "jackals_km_sm": "Jackals",
    "pigeons_km_sm": "Pigeons",
    "spur_winged_lapwings_km_sm": "Spur-winged lapwings",
    "spur_winged_lapwings": "Spur-winged lapwings",
}

# (output_key, latex_label, source_column, transform_fn)
METRICS = [
    ("reward", "Total reward", "reward", lambda s: s),
    ("monitoring_reward", "Monitoring reward", "r_dist", lambda s: s + 1),
    ("disturbance_penalty", "Disturbance penalty", "p_disturbance", lambda s: s),
]

AGENT_CSVS = {
    "ppo": "ppo_ep.csv",
    "sac": "sac_ep.csv",
    "dqn": "dqn_ep.csv",
}
RULE_BASED_CSV = "CentroidStandoff_ep.csv"

# ---------------------------
# Helpers
# ---------------------------
def resolve_root() -> Path:
    for root in ROOT_CANDIDATES:
        if root.exists():
            return root
    raise FileNotFoundError(
        "Could not find eval animal root. Tried:\n- "
        + "\n- ".join(str(p) for p in ROOT_CANDIDATES)
    )

def fmt_mean_std(series: pd.Series) -> str:
    mean = series.mean()
    std = series.std(ddof=1)
    return f"${mean:.3f} \\pm {std:.3f}$"

def read_eval_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, comment="#")
    required_cols = [source_col for _, _, source_col, _ in METRICS]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{csv_path} is missing required columns: {missing}")
    return df

def summarize_df(df: pd.DataFrame) -> dict:
    return {
        output_key: fmt_mean_std(transform_fn(df[source_col]))
        for output_key, _, source_col, transform_fn in METRICS
    }

def selection_score(df: pd.DataFrame) -> float:
    # Best eval criterion: highest mean total reward
    return float(df["reward"].mean())

def animal_display_name(animal_key: str) -> str:
    return ANIMAL_DISPLAY.get(animal_key, animal_key.replace("_", " "))

def infer_config_from_path(path: Path) -> str | None:
    s = str(path).replace("\\", "/").upper()

    patterns = [
        ("LPOI", [r"(^|[^A-Z])LPOI([^A-Z]|$)"]),
        ("POI",  [r"(^|[^A-Z])POI([^A-Z]|$)"]),
        ("CRW",  [r"(^|[^A-Z])CRW([^A-Z]|$)"]),
        ("EE",   [r"(^|[^A-Z])EE([^A-Z]|$)"]),
    ]

    for cfg, pats in patterns:
        for pat in pats:
            if re.search(pat, s):
                return cfg
    return None

def find_animal_dirs(root: Path) -> dict[str, Path]:
    found = {}
    for child in root.iterdir():
        if not child.is_dir():
            continue
        name = child.name
        if "jackals" in name:
            found["jackals_km_sm"] = child
        elif "pigeons" in name:
            found["pigeons_km_sm"] = child
        elif "spur_winged_lapwings" in name:
            found[name] = child
    return found

def collect_csvs(animal_dir: Path, filename: str) -> list[Path]:
    return sorted(animal_dir.rglob(filename))

# ---------------------------
# Scan eval directories
# ---------------------------
root = resolve_root()
animal_dirs = find_animal_dirs(root)

if not animal_dirs:
    raise FileNotFoundError(f"No animal subdirectories found under {root}")

# full results for the big table
results = {}  # results[animal][model][config] = summary dict

# keep numeric scores so we can choose the best config later
scores = {}   # scores[animal][model][config] = mean reward score

for animal_key, animal_dir in animal_dirs.items():
    results.setdefault(animal_key, {})
    scores.setdefault(animal_key, {})

    # -----------------------
    # Agent-based models
    # -----------------------
    for agent, csv_name in AGENT_CSVS.items():
        best_by_config = {}

        for csv_path in collect_csvs(animal_dir, csv_name):
            config = infer_config_from_path(csv_path)
            if config not in CONFIG_ORDER:
                continue

            df = read_eval_csv(csv_path)
            score = selection_score(df)

            current = best_by_config.get(config)
            if current is None or score > current["score"]:
                best_by_config[config] = {
                    "score": score,
                    "summary": summarize_df(df),
                }

        if best_by_config:
            results[animal_key].setdefault(agent, {})
            scores[animal_key].setdefault(agent, {})
            for config, item in best_by_config.items():
                results[animal_key][agent][config] = item["summary"]
                scores[animal_key][agent][config] = item["score"]

    # -----------------------
    # Rule-based baseline
    # -----------------------
    baseline_by_config = {}
    baseline_scores = {}

    for csv_path in collect_csvs(animal_dir, RULE_BASED_CSV):
        config = infer_config_from_path(csv_path)
        if config not in CONFIG_ORDER:
            continue
        if config in baseline_by_config:
            continue

        df = read_eval_csv(csv_path)
        baseline_by_config[config] = summarize_df(df)
        baseline_scores[config] = selection_score(df)

    if baseline_by_config:
        results[animal_key]["rule-based"] = baseline_by_config
        scores[animal_key]["rule-based"] = baseline_scores

# ---------------------------
# Build full LaTeX table
# ---------------------------
full_lines = []
full_lines.append(r"\begin{table*}[!ht]")
full_lines.append(r"\centering")
full_lines.append(r"\setlength{\tabcolsep}{4pt}")
full_lines.append(
    r"\caption{Performance comparison across animals and behavior priors for each animal, movement type, and agent. "
    r"Results are reported as mean $\pm$ standard deviation across evaluation episodes.}"
)
full_lines.append(r"\label{tab:behavior_results_full_animals}")
full_lines.append(r"\begin{tabular}{lllcccc}")
full_lines.append(r"\hline \hline")
full_lines.append(
    r"\textbf{Animal} & \textbf{Model} & \textbf{Metric} & \textbf{CRW} & \textbf{EE} & \textbf{POI} & \textbf{LPOI} \\"
)
full_lines.append(r"\hline \hline")

animal_keys_sorted = sorted(results.keys(), key=lambda k: animal_display_name(k).lower())

first_animal = True
for animal_key in animal_keys_sorted:
    animal_block = results[animal_key]
    animal_label = animal_display_name(animal_key)
    models_present = [m for m in MODEL_ORDER if m in animal_block and animal_block[m]]

    if not models_present:
        continue

    if not first_animal:
        full_lines.append(r"\midrule")
    first_animal = False

    animal_row_count = len(models_present) * len(METRICS)
    animal_row_idx = 0

    for mi, model_key in enumerate(models_present):
        model_block = animal_block[model_key]
        model_label = MODEL_LABELS[model_key]

        for metric_idx, (output_key, metric_label, _, _) in enumerate(METRICS):
            row_cells = []

            if animal_row_idx == 0:
                row_cells.append(rf"\multirow{{{animal_row_count}}}{{*}}{{{animal_label}}}")
            else:
                row_cells.append("")

            row_cells.append(model_label if metric_idx == 0 else "")
            row_cells.append(metric_label)

            for config in CONFIG_ORDER:
                value = model_block.get(config, {}).get(output_key, "--")
                row_cells.append(value)

            full_lines.append(" & ".join(row_cells) + r" \\")
            animal_row_idx += 1

        if mi < len(models_present) - 1:
            full_lines.append(r"\cmidrule(lr){2-7}")

full_lines.append(r"\hline \hline")
full_lines.append(r"\end{tabular}")
full_lines.append(r"\end{table*}")

# ---------------------------
# Build selected table
# best config among CRW / EE / POI / LPOI
# animals on columns
# ---------------------------
best_choice = {}  # best_choice[model][animal] = {"config": ..., "summary": ...}

for animal_key in animal_keys_sorted:
    best_choice.setdefault(animal_key, {})
    animal_scores = scores.get(animal_key, {})
    animal_results = results.get(animal_key, {})

    for model_key in MODEL_ORDER:
        model_scores = animal_scores.get(model_key, {})
        if not model_scores:
            continue

        best_config = max(model_scores, key=model_scores.get)
        best_choice[animal_key][model_key] = {
            "config": best_config,
            "summary": animal_results[model_key][best_config],
        }

selected_lines = []
selected_lines.append(r"\begin{table*}[!ht]")
selected_lines.append(r"\centering")
selected_lines.append(r"\setlength{\tabcolsep}{4pt}")
selected_lines.append(
    r"\caption{Performance comparison across animals and behavior priors for each animal, and agent using the best performing movement type. "
    r"Results are reported as mean $\pm$ standard deviation across evaluation episodes.}"
)
selected_lines.append(r"\label{tab:behavior_results_selected_animals}")
selected_lines.append(r"\begin{tabular}{llccc}")
selected_lines.append(r"\hline \hline")
selected_lines.append(
    r"\textbf{Model} & \textbf{Metric} & \textbf{Jackals} & \textbf{Pigeons} & \textbf{Spur-winged lapwings} \\"
)
selected_lines.append(r"\hline \hline")

animal_col_order = []
for key in animal_keys_sorted:
    label = animal_display_name(key)
    if label == "Jackals":
        animal_col_order.append(key)
for key in animal_keys_sorted:
    label = animal_display_name(key)
    if label == "Pigeons":
        animal_col_order.append(key)
for key in animal_keys_sorted:
    label = animal_display_name(key)
    if label == "Spur-winged lapwings":
        animal_col_order.append(key)

animal_col_order = list(dict.fromkeys(animal_col_order))

models_present = []
for model_key in MODEL_ORDER:
    present = any(model_key in best_choice.get(animal_key, {}) for animal_key in animal_col_order)
    if present:
        models_present.append(model_key)

for mi, model_key in enumerate(models_present):
    model_label = MODEL_LABELS[model_key]

    for metric_idx, (output_key, metric_label, _, _) in enumerate(METRICS):
        row_cells = []
        row_cells.append(model_label if metric_idx == 0 else "")
        row_cells.append(metric_label)

        for animal_key in animal_col_order:
            item = best_choice.get(animal_key, {}).get(model_key)
            if item is None:
                row_cells.append("--")
            else:
                cfg = item["config"]
                value = item["summary"][output_key]
                row_cells.append(rf"{value} ({cfg})")

        selected_lines.append(" & ".join(row_cells) + r" \\")
        selected_lines.append("")

    if mi < len(models_present) - 1:
        selected_lines.append(r"\midrule")

selected_lines.append(r"\hline \hline")
selected_lines.append(r"\end{tabular}")
selected_lines.append(r"\end{table*}")

# ---------------------------
# Print both tables
# ---------------------------
print("\n".join(full_lines))
print()
print("\n".join(selected_lines))

\begin{table*}[!ht]
\centering
\setlength{\tabcolsep}{4pt}
\caption{Performance comparison across animals and behavior priors for each animal, movement type, and agent. Results are reported as mean $\pm$ standard deviation across evaluation episodes.}
\label{tab:behavior_results_full_animals}
\begin{tabular}{lllcccc}
\hline \hline
\textbf{Animal} & \textbf{Model} & \textbf{Metric} & \textbf{CRW} & \textbf{EE} & \textbf{POI} & \textbf{LPOI} \\
\hline \hline
\multirow{12}{*}{Jackals} & Rule-based & Total reward & -- & -- & -- & $0.849 \pm 0.013$ \\
 &  & Monitoring reward & -- & -- & -- & $0.501 \pm 0.001$ \\
 &  & Disturbance penalty & -- & -- & -- & $0.188 \pm 0.006$ \\
\cmidrule(lr){2-7}
 & DQN & Total reward & -- & -- & -- & $0.914 \pm 0.008$ \\
 &  & Monitoring reward & -- & -- & -- & $0.572 \pm 0.015$ \\
 &  & Disturbance penalty & -- & -- & -- & $0.202 \pm 0.013$ \\
\cmidrule(lr){2-7}
 & PPO & Total reward & -- & -- & -- & $0.910 \pm 0.072$ \\
 &  & Monitoring reward & -- & -- & -

In [7]:
from pathlib import Path
import pandas as pd

# manifest_csv = Path("table/replay_eval_manifest_20260505_164832.csv")
# evals_root = Path("evals/replay1")

manifest_csv = Path("table/replay_eval_manifest_20260505_225338.csv")
evals_root = Path("evals/replay2")

SPECIES_ORDER = ["JACKALS", "PIGEONS", "SPUR"]
TRAINING_ORDER = ["BEST", "GPS"]
MODEL_ORDER = ["sac"]

SPECIES_LABELS = {
    "JACKALS": "Jackal",
    "PIGEONS": "Pigeon",
    "SPUR": "Spur-winged lapwing",
}

MODEL_LABELS = {
    "sac": "SAC",
}

BEST_TRAINING_LABELS = {
    "JACKALS": "LPOI",
    "PIGEONS": "LPOI",
    "SPUR": "EE",
}

# (output_key, row_label, source_column, transform)
METRICS = [
    ("reward", "Total reward", "reward", lambda s: s),
    ("monitoring_reward", "Monitoring reward", "r_dist", lambda s: s + 1),
    ("disturbance_penalty", "Disturbance penalty", "p_disturbance", lambda s: s),
]

def fmt_mean_std(series: pd.Series) -> str:
    return f"${series.mean():.3f} \\pm {series.std(ddof=1):.3f}$"

manifest = pd.read_csv(manifest_csv)

results = {}

for _, row in manifest.iterrows():
    cfg = str(row["config"]).strip()
    agent = str(row["agent"]).strip().lower()
    eval_name = str(row["eval_name"]).strip()

    if cfg.startswith("JACKALS"):
        species = "JACKALS"
    elif cfg.startswith("PIGEONS"):
        species = "PIGEONS"
    elif cfg.startswith("SPUR"):
        species = "SPUR"
    else:
        continue

    training_data = "GPS" if "GPS" in cfg else "BEST"

    ep_csv = evals_root / eval_name / f"{agent}_ep.csv"
    if not ep_csv.exists():
        raise FileNotFoundError(ep_csv)

    df = pd.read_csv(ep_csv, comment="#")

    results.setdefault(training_data, {})
    results[training_data].setdefault(agent, {})
    results[training_data][agent][species] = {
        out_key: fmt_mean_std(transform(df[source_col]))
        for out_key, _, source_col, transform in METRICS
    }

lines = []
lines.append(r"\begin{table}[!ht]")
lines.append(r"\centering")
lines.append(r"\setlength{\tabcolsep}{4pt}")
lines.append(r"\caption{Replay evaluation performance across species, training data, and models. Results are reported as mean $\pm$ standard deviation across 100 evaluation episodes.}")
lines.append(r"\label{tab:replay_results_all_species}")
lines.append(r"\begin{tabular}{llcccc}")
lines.append(r"\hline \hline")
lines.append(r"\textbf{Training} & \textbf{Model} & \textbf{Metric} & \textbf{Jackal} & \textbf{Pigeon} & \textbf{Spur-winged lapwing} \\")
lines.append(r"\hline \hline")

row_blocks = []
for training_data in TRAINING_ORDER:
    training_label = {
        "JACKALS": "GPS" if training_data == "GPS" else BEST_TRAINING_LABELS["JACKALS"],
        "PIGEONS": "GPS" if training_data == "GPS" else BEST_TRAINING_LABELS["PIGEONS"],
        "SPUR": "GPS" if training_data == "GPS" else BEST_TRAINING_LABELS["SPUR"],
    }

    # single label for the row block
    block_label = "GPS" if training_data == "GPS" else "Best-fit"

    for agent in MODEL_ORDER:
        for metric_idx, (out_key, metric_label, _, _) in enumerate(METRICS):
            training_cell = rf"\multirow{{6}}{{*}}{{{block_label}}}" if (agent == MODEL_ORDER[0] and metric_idx == 0) else ""
            model_cell = rf"\multirow{{3}}{{*}}{{{MODEL_LABELS[agent]}}}" if metric_idx == 0 else ""

            row = [
                training_cell,
                model_cell,
                metric_label,
            ]

            for species in SPECIES_ORDER:
                value = results.get(training_data, {}).get(agent, {}).get(species, {}).get(out_key, "--")
                row.append(value)

            lines.append(" & ".join(row) + r" \\")
        lines.append("")

    if training_data != TRAINING_ORDER[-1]:
        lines.append(r"\midrule")

lines.append(r"\hline \hline")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)
print(latex_table)

\begin{table}[!ht]
\centering
\setlength{\tabcolsep}{4pt}
\caption{Replay evaluation performance across species, training data, and models. Results are reported as mean $\pm$ standard deviation across 100 evaluation episodes.}
\label{tab:replay_results_all_species}
\begin{tabular}{llcccc}
\hline \hline
\textbf{Training} & \textbf{Model} & \textbf{Metric} & \textbf{Jackal} & \textbf{Pigeon} & \textbf{Spur-winged lapwing} \\
\hline \hline
\multirow{6}{*}{Best-fit} & \multirow{3}{*}{SAC} & Total reward & $0.930 \pm 0.006$ & $0.876 \pm 0.051$ & $0.930 \pm 0.010$ \\
 &  & Monitoring reward & $0.582 \pm 0.011$ & $0.523 \pm 0.021$ & $0.568 \pm 0.006$ \\
 &  & Disturbance penalty & $0.200 \pm 0.012$ & $0.175 \pm 0.014$ & $0.185 \pm 0.009$ \\

\midrule
\multirow{6}{*}{GPS} & \multirow{3}{*}{SAC} & Total reward & $0.931 \pm 0.008$ & $0.860 \pm 0.128$ & $0.891 \pm 0.140$ \\
 &  & Monitoring reward & $0.565 \pm 0.009$ & $0.497 \pm 0.021$ & $0.525 \pm 0.025$ \\
 &  & Disturbance penalty & $0.185 \p